# 04 · Modelling, Cost-Based Threshold and Risk Segments

Compares three models, picks the best on validation, chooses a cost-optimal threshold, builds four risk segments, and evaluates **once** on the untouched test set.

## 1. Imports

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

## 2. Load data

In [5]:
path = '../data/'

X_train = pd.read_csv(path + 'X_train.csv')
X_val = pd.read_csv(path + 'X_val.csv')
X_test = pd.read_csv(path + 'X_test.csv')

y_train = pd.read_csv(path + 'y_train.csv').squeeze()
y_val = pd.read_csv(path + 'y_val.csv').squeeze()
y_test = pd.read_csv(path + 'y_test.csv').squeeze()

ids_test = pd.read_csv(path + 'ids_test.csv').squeeze()

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

Train: (215257, 37)  Val: (46127, 37)  Test: (46127, 37)


## 3. Two feature sets

Logistic regression and random forest get the imputed EXT_SOURCE columns, since they can't accept missing values. XGBoost gets the raw versions with real NaNs, since it can learn from missingness.

In [7]:
raw_cols = ['EXT_SOURCE_1_raw', 'EXT_SOURCE_2_raw', 'EXT_SOURCE_3_raw']
imputed_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

def make_standard_features(X):
    return X.drop(columns=raw_cols)

def make_xgb_features(X):
    X_xgb = X.drop(columns=imputed_cols)
    return X_xgb.rename(columns=dict(zip(raw_cols, imputed_cols)))

X_train_std, X_val_std, X_test_std = (make_standard_features(d) for d in (X_train, X_val, X_test))
X_train_xgb, X_val_xgb, X_test_xgb = (make_xgb_features(d) for d in (X_train, X_val, X_test))

print("Standard feature set:", X_train_std.shape[1], "columns   XGBoost feature set:", X_train_xgb.shape[1], "columns")

Standard feature set: 34 columns   XGBoost feature set: 34 columns


## 4. Models

`class_weight='balanced'` and `scale_pos_weight` correct for the ~8% default rate. Logistic regression is wrapped in a Pipeline, so the scaler is fitted on training data only, including inside cross-validation.

In [9]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
xgb_weight = neg / pos

lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, scale_pos_weight=xgb_weight,
    eval_metric='logloss', random_state=42
)

## 5. Cross-validation on train only (5-fold ROC-AUC)

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv = cross_val_score(lr_model, X_train_std, y_train, cv=cv, scoring='roc_auc')
rf_cv = cross_val_score(rf_model, X_train_std, y_train, cv=cv, scoring='roc_auc')
xgb_cv = cross_val_score(xgb_model, X_train_xgb, y_train, cv=cv, scoring='roc_auc')

print("CV ROC-AUC (train only, 5-fold)")
print("Logistic Regression:", round(lr_cv.mean(), 4), "+/-", round(lr_cv.std(), 4))
print("Random Forest      :", round(rf_cv.mean(), 4), "+/-", round(rf_cv.std(), 4))
print("XGBoost            :", round(xgb_cv.mean(), 4), "+/-", round(xgb_cv.std(), 4))

CV ROC-AUC (train only, 5-fold)
Logistic Regression: 0.7355 +/- 0.0055
Random Forest      : 0.7383 +/- 0.0063
XGBoost            : 0.7436 +/- 0.0057


## 6. Fit on train, score on validation

In [13]:
lr_model.fit(X_train_std, y_train)
rf_model.fit(X_train_std, y_train)
xgb_model.fit(X_train_xgb, y_train)

val_probs = {
    'Logistic Regression': lr_model.predict_proba(X_val_std)[:, 1],
    'Random Forest': rf_model.predict_proba(X_val_std)[:, 1],
    'XGBoost': xgb_model.predict_proba(X_val_xgb)[:, 1],
}

print("Validation performance")
for name, probs in val_probs.items():
    auc = roc_auc_score(y_val, probs)
    pr_auc = average_precision_score(y_val, probs)
    print(f"{name:22s} ROC-AUC: {auc:.4f}   PR-AUC: {pr_auc:.4f}")

best_model_name = max(val_probs, key=lambda k: roc_auc_score(y_val, val_probs[k]))
best_val_probs = val_probs[best_model_name]
print("\nBest model on validation:", best_model_name)

Validation performance
Logistic Regression    ROC-AUC: 0.7439   PR-AUC: 0.2248
Random Forest          ROC-AUC: 0.7453   PR-AUC: 0.2294
XGBoost                ROC-AUC: 0.7506   PR-AUC: 0.2411

Best model on validation: XGBoost


## 7. Cost-sensitive threshold (chosen on validation)

A missed default (false negative) is assumed to cost the average training loan amount, and a wrongly rejected good borrower (false positive) 10% of that. These are illustrative business assumptions, so the threshold is also recalculated under other cost ratios.

In [15]:
fn_cost = X_train['loan_amount'].mean()
fp_cost = fn_cost * 0.10
print(f"FN cost: {fn_cost:,.0f}   FP cost: {fp_cost:,.0f}  (10:1 ratio, illustrative)")

def cost_at_threshold(y_true, probs, threshold, fn_cost, fp_cost):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return fn * fn_cost + fp * fp_cost

thresholds = np.arange(0.01, 1.00, 0.01)
costs = [cost_at_threshold(y_val, best_val_probs, t, fn_cost, fp_cost) for t in thresholds]
optimal_threshold = thresholds[np.argmin(costs)]
print("Optimal threshold (from validation):", round(optimal_threshold, 3))

print("\nSensitivity to the FN:FP cost ratio")
for ratio in [5, 10, 15, 20]:
    fp_cost_ratio = fn_cost / ratio
    costs_ratio = [cost_at_threshold(y_val, best_val_probs, t, fn_cost, fp_cost_ratio) for t in thresholds]
    best_t = thresholds[np.argmin(costs_ratio)]
    print(f"Ratio {ratio}:1  ->  optimal threshold {best_t:.2f}")

FN cost: 598,076   FP cost: 59,808  (10:1 ratio, illustrative)
Optimal threshold (from validation): 0.54

Sensitivity to the FN:FP cost ratio
Ratio 5:1  ->  optimal threshold 0.67
Ratio 10:1  ->  optimal threshold 0.54
Ratio 15:1  ->  optimal threshold 0.40
Ratio 20:1  ->  optimal threshold 0.30


## 8. Risk segment boundaries (from validation)

- **A** (Auto Approve): below the median score
- **B** (Standard Review): median to 60th percentile
- **C** (Manual Review): 60th percentile to the optimal threshold
- **D** (High-Risk Review): at or above the optimal threshold

In [17]:
median_cut = np.median(best_val_probs)
p60_cut = np.percentile(best_val_probs, 60)
print("Segment boundaries (from validation):")
print("median:", round(median_cut, 3), " 60th pct:", round(p60_cut, 3),
      " optimal threshold:", round(optimal_threshold, 3))

if optimal_threshold <= p60_cut:
    print("WARNING: optimal threshold is below the 60th percentile - segment C will be empty")

def assign_segment(prob):
    if prob < median_cut:
        return 'A'
    elif prob < p60_cut:
        return 'B'
    elif prob < optimal_threshold:
        return 'C'
    else:
        return 'D'

Segment boundaries (from validation):
median: 0.374  60th pct: 0.437  optimal threshold: 0.54


## 9. Final evaluation on test (done once)

The output is a **risk score**, not a probability: class weighting pushes scores upward, so the average score is far above the real default rate. Scores rank borrowers correctly but are not calibrated.

In [19]:
best_models = {'Logistic Regression': (lr_model, X_test_std),
               'Random Forest': (rf_model, X_test_std),
               'XGBoost': (xgb_model, X_test_xgb)}

final_model, final_X_test = best_models[best_model_name]
test_probs = final_model.predict_proba(final_X_test)[:, 1]
test_preds = (test_probs >= optimal_threshold).astype(int)

print("Final test results -", best_model_name)
print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))
print("PR-AUC :", round(average_precision_score(y_test, test_probs), 4))
print(classification_report(y_test, test_preds))

final_cost = cost_at_threshold(y_test, test_probs, optimal_threshold, fn_cost, fp_cost)
baseline_cost = y_test.sum() * fn_cost
cost_reduction_pct = round((1 - final_cost / baseline_cost) * 100, 1)

print(f"Baseline cost (no model): {baseline_cost:,.0f}")
print(f"Model cost at threshold : {final_cost:,.0f}")
print(f"Cost reduction          : {cost_reduction_pct}%")
print(f"\nAverage risk score: {test_probs.mean():.3f}   Actual default rate: {y_test.mean():.3f}")

Final test results - XGBoost
ROC-AUC: 0.7526
PR-AUC : 0.235
              precision    recall  f1-score   support

           0       0.96      0.77      0.85     42403
           1       0.18      0.60      0.28      3724

    accuracy                           0.75     46127
   macro avg       0.57      0.68      0.57     46127
weighted avg       0.89      0.75      0.80     46127

Baseline cost (no model): 2,227,235,827
Model cost at threshold : 1,490,286,314
Cost reduction          : 33.1%

Average risk score: 0.399   Actual default rate: 0.081


## 10. Model summary

In [21]:
model_summary = pd.DataFrame([
    {'Metric': 'Total Borrowers', 'Value': len(y_test)},
    {'Metric': 'Overall Default Rate %', 'Value': round(y_test.mean() * 100, 2)},
    {'Metric': 'Best Model', 'Value': best_model_name},
    {'Metric': 'ROC-AUC', 'Value': round(roc_auc_score(y_test, test_probs), 4)},
    {'Metric': 'PR-AUC', 'Value': round(average_precision_score(y_test, test_probs), 4)},
    {'Metric': 'Optimal Threshold', 'Value': optimal_threshold},
    {'Metric': 'Cost Reduction %', 'Value': cost_reduction_pct},
    {'Metric': 'Baseline Cost', 'Value': round(baseline_cost)},
    {'Metric': 'Optimized Cost', 'Value': round(final_cost)},
    {'Metric': 'Cost Saved', 'Value': round(baseline_cost - final_cost)},
])

model_summary.to_csv(path + 'model_summary.csv', index=False)
model_summary

,Metric,Value
0,Total Borrowers,46127
1,Overall Default Rate %,8.07
2,Best Model,XGBoost
3,ROC-AUC,0.7526
4,PR-AUC,0.235
5,Optimal Threshold,0.54
6,Cost Reduction %,33.1
7,Baseline Cost,2227235827
8,Optimized Cost,1490286314
9,Cost Saved,736949513


## 11. Feature importance (XGBoost, gain-based)

In [23]:
importances = pd.Series(xgb_model.feature_importances_, index=X_train_xgb.columns)
print("Top 15 XGBoost features")
print(importances.sort_values(ascending=False).head(15).round(4))

Top 15 XGBoost features
EXT_SOURCE_MEAN                            0.2494
employment_type_Pensioner                  0.0845
education_Higher education                 0.0590
gender                                     0.0450
education_Secondary / secondary special    0.0449
owns_car                                   0.0389
employment_years                           0.0312
family_status_Married                      0.0294
EXT_SOURCE_3                               0.0261
loan_amount                                0.0258
region_risk_rating                         0.0255
EXT_SOURCE_1                               0.0247
employment_type_State servant              0.0236
age_years                                  0.0231
annual_repayment                           0.0228
dtype: float32


## 12. Test predictions and risk segment summary

Default rates should rise steadily from segment A to D.

In [25]:
predictions = pd.DataFrame({
    'SK_ID_CURR': ids_test,
    'Risk_Score': test_probs,
    'Risk_Segment': [assign_segment(p) for p in test_probs],
    'Actual_Default': y_test.values
})

predictions.to_csv(path + 'best_model_predictions.csv', index=False)

segment_summary = (
    predictions.groupby('Risk_Segment')['Actual_Default']
    .agg(borrowers='size', defaults='sum')
    .reindex(['A', 'B', 'C', 'D'], fill_value=0)
)
segment_summary['default_rate_pct'] = (segment_summary['defaults'] / segment_summary['borrowers'] * 100).round(2)
segment_summary['pct_of_borrowers'] = (segment_summary['borrowers'] / len(predictions) * 100).round(1)
segment_summary['pct_of_defaults'] = (segment_summary['defaults'] / predictions['Actual_Default'].sum() * 100).round(1)

segment_summary.to_csv(path + 'segment_summary.csv')
segment_summary

,borrowers,defaults,default_rate_pct,pct_of_borrowers,pct_of_defaults
Risk_Segment,,,,,
A,22901,665,2.90,49.6,17.9
B,4736,290,6.12,10.3,7.8
C,6304,541,8.58,13.7,14.5
D,12186,2228,18.28,26.4,59.8
